# 🩺 Second Opinion: Agentic Multi-Model Medical AI Pipeline

### HAI-DEF Pipeline — Benchmarking MedGemma 4B on Real Medical Images

| | |
|---|---|
| 🔴 **Live App** | [gen-lang-client-0003791133.web.app](https://gen-lang-client-0003791133.web.app) |
| 💻 **GitHub** | [TrendpilotAI/Second-Opinion](https://github.com/TrendpilotAI/Second-Opinion) |
| 🏆 **Track** | Agentic Workflow Prize |
| 👤 **Author** | Nathan Stevenson · [ForwardLane.com](https://forwardlane.com) |

---

**This notebook runs MedGemma 4B on real medical images and measures actual performance.** Every chart uses data from live inference — no simulated or fabricated results.

In [1]:
# ============================================================
# Setup: Install deps, load MedGemma 4B on T4
# ============================================================

!pip install -q transformers accelerate bitsandbytes pillow

import torch
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.figsize': (12, 5)})

COLORS = {
    'medgemma': '#4285F4', 'gemini': '#34A853',
    'flash': '#FBBC04', 'accent': '#EA4335', 'gray': '#5f6368'
}

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
GPU: Tesla P100-PCIE-16GB


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
# ============================================================
# Load MedGemma 4B-IT (INT4 quantized)
# ============================================================

from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
load_time = time.time() - t0
vram_gb = torch.cuda.memory_allocated() / 1024**3

print(f"✅ MedGemma 4B loaded in {load_time:.1f}s | VRAM: {vram_gb:.1f} GB | INT4")

In [ ]:
# ============================================================
# Inference helper
# ============================================================

def run_medgemma(image, prompt, max_tokens=1024):
    """Run MedGemma inference. Returns response + metrics."""
    t0 = time.time()
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt}
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    response = processor.decode(out[0][input_len:], skip_special_tokens=True)
    latency = time.time() - t0
    return {"response": response, "latency": round(latency, 2),
            "in_tokens": input_len, "out_tokens": int(out.shape[-1] - input_len)}

def load_img(url):
    return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

print("✅ Ready")

## 1. Benchmark: MedGemma 4B Across Medical Image Types

We test MedGemma on **5 different medical image types** from public sources to measure latency, output quality, and cost across specialties.

In [ ]:
# ============================================================
# Load 5 real medical images from public sources
# All images are public domain or CC-licensed
# ============================================================

BUCKET = "gen-lang-client-0003791133.firebasestorage.app"

test_images = {
    "Chest X-ray": {
        "url": "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png",
        "prompt": "Analyze this chest X-ray. List all findings with anatomical locations and confidence levels (HIGH/MEDIUM/LOW). Include differential diagnosis.",
        "specialty": "Radiology",
    },
    "Skin Lesion": {
        "url": "https://upload.wikimedia.org/wikipedia/commons/6/6c/Melanoma.jpg",
        "prompt": "Analyze this skin lesion. Perform ABCDE assessment. List differential diagnosis ranked by likelihood. Classify risk: benign/suspicious/urgent.",
        "specialty": "Dermatology",
    },
    "Wrist X-ray": {
        "url": f"https://storage.googleapis.com/{BUCKET}/demo/sample_case/media/images/sample_xray_1.jpg",
        "prompt": "Analyze this wrist X-ray. Assess bone alignment, fractures, joint spaces. Is surgery indicated or could conservative treatment work?",
        "specialty": "Orthopedics",
    },
    "Retinal Image": {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4c/Fundus_photograph_of_normal_right_eye.jpg/640px-Fundus_photograph_of_normal_right_eye.jpg",
        "prompt": "Analyze this fundus photograph. Assess optic disc, macula, vessels, and retina. Note any pathological findings.",
        "specialty": "Ophthalmology",
    },
    "Brain MRI": {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/Mri_brain_side.jpg/463px-Mri_brain_side.jpg",
        "prompt": "Analyze this brain MRI. Assess for mass lesions, midline shift, ventricle size, white matter changes, and any abnormalities.",
        "specialty": "Neurology",
    },
}

# Load all images
print("📥 Loading medical images...")
for name, info in test_images.items():
    try:
        info["image"] = load_img(info["url"])
        print(f"   ✅ {name}: {info['image'].size}")
    except Exception as e:
        print(f"   ❌ {name}: {e}")
        info["image"] = None

# Display all test images
valid = {k: v for k, v in test_images.items() if v["image"] is not None}
n = len(valid)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
if n == 1: axes = [axes]
for ax, (name, info) in zip(axes, valid.items()):
    ax.imshow(info["image"], cmap='gray' if 'X-ray' in name or 'MRI' in name else None)
    ax.set_title(f"{name}\n({info['specialty']})", fontsize=10)
    ax.axis('off')
plt.suptitle('Test Images — All from Public Sources', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Run MedGemma on ALL test images — REAL INFERENCE
# ============================================================

benchmark_results = {}

for name, info in valid.items():
    print(f"\n🔬 Analyzing: {name} ({info['specialty']})...")
    result = run_medgemma(info["image"], info["prompt"])
    benchmark_results[name] = {
        **result,
        "specialty": info["specialty"],
        "response_length": len(result["response"]),
    }
    print(f"   ⏱️  {result['latency']}s | {result['out_tokens']} tokens | {len(result['response'])} chars")
    print(f"   Preview: {result['response'][:200]}...")

print(f"\n✅ Completed {len(benchmark_results)} analyses")

In [ ]:
# ============================================================
# Figure 1: REAL Performance Metrics Across Specialties
# Every number comes from the inference runs above
# ============================================================

names = list(benchmark_results.keys())
latencies = [benchmark_results[n]['latency'] for n in names]
out_tokens = [benchmark_results[n]['out_tokens'] for n in names]
resp_lens = [benchmark_results[n]['response_length'] for n in names]

T4_COST_PER_SEC = 0.000164  # Modal T4 pricing
costs = [lat * T4_COST_PER_SEC for lat in latencies]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
colors = [COLORS['medgemma'], COLORS['gemini'], COLORS['accent'], COLORS['flash'], COLORS['gray']][:len(names)]

# 1a: Latency
bars = axes[0].bar(range(len(names)), latencies, color=colors, alpha=0.85)
for i, v in enumerate(latencies):
    axes[0].text(i, v + max(latencies)*0.02, f'{v:.1f}s', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('Seconds')
axes[0].set_title('1a: Inference Latency', fontweight='bold')

# 1b: Output tokens
bars = axes[1].bar(range(len(names)), out_tokens, color=colors, alpha=0.85)
for i, v in enumerate(out_tokens):
    axes[1].text(i, v + max(out_tokens)*0.02, str(v), ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('Tokens')
axes[1].set_title('1b: Output Length', fontweight='bold')

# 1c: Cost
bars = axes[2].bar(range(len(names)), [c*100 for c in costs], color=colors, alpha=0.85)
for i, c in enumerate(costs):
    axes[2].text(i, c*100 + max(costs)*100*0.02, f'${c:.4f}', ha='center', fontsize=8, fontweight='bold')
axes[2].set_xticks(range(len(names)))
axes[2].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[2].set_ylabel('Cost (cents)')
axes[2].set_title('1c: GPU Cost (Modal T4)', fontweight='bold')

# 1d: Response length
bars = axes[3].bar(range(len(names)), resp_lens, color=colors, alpha=0.85)
for i, v in enumerate(resp_lens):
    axes[3].text(i, v + max(resp_lens)*0.02, str(v), ha='center', fontsize=8, fontweight='bold')
axes[3].set_xticks(range(len(names)))
axes[3].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[3].set_ylabel('Characters')
axes[3].set_title('1d: Response Detail', fontweight='bold')

plt.suptitle('Figure 1: MedGemma 4B Real Performance — Measured on This GPU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 Averages across {len(names)} medical image types:')
print(f'   Latency: {np.mean(latencies):.1f}s ± {np.std(latencies):.1f}s')
print(f'   Output:  {np.mean(out_tokens):.0f} tokens ± {np.std(out_tokens):.0f}')
print(f'   Cost:    ${np.mean(costs):.4f} ± ${np.std(costs):.4f} per analysis')

## 2. Full Inference Outputs

Here are MedGemma's **complete, unedited responses** for each medical image. These are the raw model outputs — not cherry-picked or modified.

In [ ]:
# ============================================================
# Print FULL unedited MedGemma outputs for every test image
# ============================================================

for name, result in benchmark_results.items():
    print(f"\n{'='*70}")
    print(f"  {name.upper()} ({result['specialty']})")
    print(f"  Latency: {result['latency']}s | Tokens: {result['in_tokens']}→{result['out_tokens']}")
    print(f"{'='*70}\n")
    print(result['response'])
    print(f"\n{'─'*70}")

## 3. Agentic Pipeline Architecture

Second Opinion's production pipeline orchestrates 4 agents. Each makes autonomous decisions:

In [ ]:
# ============================================================
# Figure 2: Pipeline Architecture (from actual codebase)
# Source: functions/src/pipeline.ts
# ============================================================

fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)
ax.axis('off')

# Pipeline stages — these match pipeline.ts exactly
stages = [
    {'name': 'Patient\nInput', 'x': 0.3, 'y': 4, 'w': 1.8, 'h': 1.8, 'color': '#e3f2fd', 'border': '#1a73e8'},
    {'name': 'Step 1\nMedSigLIP-448\nImage Classification', 'x': 2.8, 'y': 5.5, 'w': 2.3, 'h': 1.8, 'color': '#fff3e0', 'border': '#e8710a'},
    {'name': 'Step 2\nGemini Flash\nComplexity Triage', 'x': 2.8, 'y': 2.5, 'w': 2.3, 'h': 1.8, 'color': '#fff9c4', 'border': COLORS['flash']},
    {'name': 'Step 3\nMedGemma 4B\nor 27B', 'x': 6.2, 'y': 4, 'w': 2.3, 'h': 2.2, 'color': '#e8f5e9', 'border': COLORS['gemini']},
    {'name': 'Step 4\nGemini Pro\nTranslation', 'x': 9.5, 'y': 4, 'w': 2.3, 'h': 2.2, 'color': '#f3e5f5', 'border': '#9c27b0'},
    {'name': 'Patient\nReport', 'x': 12.8, 'y': 4, 'w': 2.2, 'h': 1.8, 'color': '#fce4ec', 'border': COLORS['accent']},
]

for s in stages:
    rect = mpatches.FancyBboxPatch(
        (s['x'], s['y'] - s['h']/2), s['w'], s['h'],
        boxstyle='round,pad=0.12', facecolor=s['color'], edgecolor=s['border'], linewidth=2.5
    )
    ax.add_patch(rect)
    ax.text(s['x'] + s['w']/2, s['y'], s['name'],
            ha='center', va='center', fontsize=8.5, fontweight='bold')

# Arrows
ap = dict(arrowstyle='->', lw=2, color='#444')
# Input splits to parallel classification + triage
ax.annotate('', xy=(2.8, 5.8), xytext=(2.1, 4.5), arrowprops=ap)
ax.annotate('', xy=(2.8, 3.2), xytext=(2.1, 3.8), arrowprops=ap)
ax.text(2.3, 5.2, 'parallel', fontsize=7, color='#666', fontstyle='italic', rotation=45)
# Classification + Triage → MedGemma
ax.annotate('', xy=(6.2, 4.8), xytext=(5.1, 5.8), arrowprops=ap)
ax.annotate('', xy=(6.2, 3.8), xytext=(5.1, 3.2), arrowprops={**ap, 'color': COLORS['flash']})
ax.text(5.5, 3.0, 'simple→4B\ncomplex→27B', fontsize=7, color=COLORS['flash'], ha='center')
# MedGemma → Translation
ax.annotate('', xy=(9.5, 4.5), xytext=(8.5, 4.5), arrowprops=ap)
# Translation → Report
ax.annotate('', xy=(12.8, 4.5), xytext=(11.8, 4.5), arrowprops=ap)

# Fallback chain annotation
ax.text(7.3, 2.0, 'Fallback: 27B → 4B → Gemini Pro', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff3e0', edgecolor='#e8710a'),
        ha='center')

# Code reference
ax.text(8, 0.8, 'Source: functions/src/pipeline.ts — runAgenticPipeline()',
        fontsize=8, ha='center', color='#666', fontstyle='italic')

ax.set_title('Figure 2: HAI-DEF Agentic Pipeline (Production Architecture)',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 4. Cost Analysis: Real Numbers

We measure actual GPU costs from our benchmark runs, not estimates.

In [ ]:
# ============================================================
# Figure 3: Real Cost Analysis
# GPU costs are MEASURED from benchmark latencies above
# API costs are from published Gemini pricing
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3a: Cost breakdown per pipeline stage (measured + published)
avg_medgemma_cost = np.mean(costs)
pipeline_costs = {
    'MedSigLIP\n(Classification)': 0.0001,     # Free/negligible (stub in current deploy)
    'Gemini Flash\n(Triage)': 0.001,            # Published: ~$0.075/1M input tokens
    f'MedGemma 4B\n(Analysis)': avg_medgemma_cost,  # MEASURED from our runs
    'Gemini Pro\n(Translation)': 0.004,         # Published: ~$1.25/1M input tokens
}

stage_names = list(pipeline_costs.keys())
stage_costs = list(pipeline_costs.values())
stage_colors = ['#e8710a', COLORS['flash'], COLORS['medgemma'], '#9c27b0']

bars = axes[0].bar(range(len(stage_names)), [c*100 for c in stage_costs],
                   color=stage_colors, alpha=0.85, edgecolor='white')
for i, c in enumerate(stage_costs):
    label = f'${c:.4f}' if c < 0.01 else f'${c:.3f}'
    measured = ' (measured)' if 'MedGemma' in stage_names[i] else ' (published)'
    axes[0].text(i, c*100 + max(stage_costs)*100*0.03, label + measured,
                ha='center', fontsize=8, fontweight='bold')

axes[0].set_xticks(range(len(stage_names)))
axes[0].set_xticklabels(stage_names, fontsize=9)
axes[0].set_ylabel('Cost (cents)')
axes[0].set_title('3a: Per-Stage Pipeline Cost', fontweight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

total_pipeline = sum(stage_costs)
axes[0].text(0.5, 0.92, f'Total pipeline: ${total_pipeline:.4f}',
            transform=axes[0].transAxes, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor=COLORS['gemini']))

# 3b: Comparison with alternatives
alternatives = {
    f'Second Opinion\n(${total_pipeline:.3f})': total_pipeline,
    'Human Second\nOpinion ($350)': 350,
    'Specialist Visit\n($200)': 200,
    'Telehealth\n($75)': 75,
}

alt_names = list(alternatives.keys())
alt_costs = list(alternatives.values())
alt_colors = [COLORS['gemini'], COLORS['gray'], COLORS['gray'], COLORS['gray']]

bars = axes[1].barh(alt_names, alt_costs, color=alt_colors, alpha=0.85)
for bar, val in zip(bars, alt_costs):
    label = f'${val:.3f}' if val < 1 else f'${val:.0f}'
    axes[1].text(bar.get_width() + max(alt_costs)*0.02, bar.get_y() + bar.get_height()/2,
                label, va='center', fontsize=10, fontweight='bold')
axes[1].set_xscale('log')
axes[1].set_xlabel('Cost (USD, log scale)')
axes[1].set_title('3b: Cost vs Alternatives', fontweight='bold')
axes[1].invert_yaxis()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Figure 3: Real Cost Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

ratio = 350 / total_pipeline
print(f'\n💰 Second Opinion is {ratio:,.0f}x cheaper than a human second opinion')
print(f'   MedGemma GPU cost (measured): ${avg_medgemma_cost:.4f}')
print(f'   Full pipeline cost: ${total_pipeline:.4f}')

## 5. Qualitative Analysis: What MedGemma Gets Right

Rather than fabricate accuracy scores, we do a **qualitative review** of MedGemma's outputs against known characteristics of each test image.

In [ ]:
# ============================================================
# Qualitative scoring: human review of MedGemma outputs
# Scored by the author (not MedGemma scoring itself)
# ============================================================

# These scores are HUMAN JUDGMENTS of the model outputs shown above.
# Scale: 1 (poor) to 5 (excellent)
# Criteria:
#   Specificity: Does it name specific anatomical structures/findings?
#   Structure: Is the response well-organized and systematic?
#   Safety: Does it include appropriate caveats and disclaimers?
#   Actionability: Does it give useful next steps?

# NOTE: These are subjective author assessments, not ground-truth labels.
# Real clinical validation would require radiologist review.

print("⚠️  IMPORTANT: The scores below are the author's subjective assessment")
print("   of MedGemma's output quality. They are NOT validated against")
print("   radiologist ground truth. Real clinical validation is future work.")
print()
print("Scoring each MedGemma output on 4 criteria (1-5 scale):")
print("─" * 65)
print(f"{'Image':<16} {'Specificity':>12} {'Structure':>10} {'Safety':>8} {'Actionable':>11}")
print("─" * 65)

# Author will fill these after reviewing actual outputs
# For now, placeholder instruction:
print("[Scores to be filled after reviewing MedGemma outputs above]")
print()
print("This is intentionally left for human review rather than")
print("auto-generating fake quality scores.")

## 6. Production App: What Wraps the Pipeline

The MedGemma pipeline demonstrated above powers a **production web application** deployed at [gen-lang-client-0003791133.web.app](https://gen-lang-client-0003791133.web.app).

### Architecture
```
┌────────────────────────────────────────────────────────────┐
│  Firebase Hosting (CDN)              React 19 + TypeScript │
│  ├── Auth screen with narrative carousel                   │
│  ├── Conversational intake (Gemini Flash)                  │
│  ├── Multi-format file upload (HEIC, DICOM, JPEG, etc.)   │
│  └── 5-view analysis dashboard                             │
├────────────────────────────────────────────────────────────┤
│  Cloud Functions v2 (Node.js 20)     Agentic Orchestrator  │
│  ├── pipeline.ts — runAgenticPipeline()                    │
│  ├── medgemma.ts — MedGemma 4B/27B with fallback chain    │
│  └── analysis.ts — Structured clinical analysis            │
├────────────────────────────────────────────────────────────┤
│  Self-Hosted Inference (Modal)       MedGemma + vLLM       │
│  ├── medgemma_4b.py — T4 GPU, INT4, scale-to-zero         │
│  └── medgemma_27b.py — A100 40GB, AWQ, on-demand          │
└────────────────────────────────────────────────────────────┘
```

### Key Components (40 React components)

| Component | Purpose |
|-----------|--------|
| `Auth.tsx` | Narrative carousel with Tracey's story |
| `PatientChat.tsx` | Conversational symptom intake |
| `FileUploader.tsx` | Multi-format medical file upload |
| `AnalysisDashboard.tsx` | 5-view results display |
| `AgenticPipeline.tsx` | Real-time pipeline status visualization |
| `ProcessingLogWindow.tsx` | Live processing logs |
| `DoctorQuestions.tsx` | AI-generated questions for your doctor |
| `MedicalDisclaimer.tsx` | Scroll-to-accept, 3 locales |
| `GuidedDemo.tsx` | First-time user walkthrough |
| `SpecialistDirectory.tsx` | Location-based specialist matching |

Full source: [github.com/TrendpilotAI/Second-Opinion](https://github.com/TrendpilotAI/Second-Opinion)

In [ ]:
# ============================================================
# Show the narrative media from the production app
# ============================================================

stages = [
    ("The Diagnosis", f"https://storage.googleapis.com/{BUCKET}/narrative-media/stage_0_diagnosis_v1.jpg"),
    ("Second Opinion", f"https://storage.googleapis.com/{BUCKET}/narrative-media/stage_1_second_opinion_v1.jpg"),
    ("Mia's Analysis", f"https://storage.googleapis.com/{BUCKET}/narrative-media/stage_2_mia_analysis_v1.jpg"),
    ("Right Decision", f"https://storage.googleapis.com/{BUCKET}/narrative-media/stage_3_right_decision_v1.jpg"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, (title, url) in enumerate(stages):
    try:
        img = load_img(url)
        axes[i].imshow(img)
    except:
        axes[i].text(0.5, 0.5, 'Load failed', ha='center', va='center')
    axes[i].set_title(title, fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("Tracey's Story — Auth Screen Narrative (Production App)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("These Disney/Pixar-style illustrations tell Tracey's story on the login screen.")
print("Generated via AI image models and served from Firebase Storage.")

## 7. Safety & Compliance

| Control | Status | Code Reference |
|---------|--------|---------------|
| Medical Disclaimers | ✅ Implemented | `MedicalDisclaimer.tsx` — scroll-to-accept, 3 locales |
| HIPAA Audit Logging | ✅ Implemented | `utils/auditLog.ts` — structured Firestore trail |
| Authentication | ✅ Implemented | Firebase Auth (email + Google OAuth) |
| Access Control | ✅ Implemented | `storage.rules` — user-scoped, admin-only writes |
| No PHI Storage | ✅ Architecture | Ephemeral processing by design |
| FHIR R4 | 🔶 Interfaces only | Typed interfaces in `types.ts`, not full endpoints |

Full documentation: [HIPAA_COMPLIANCE.md](https://github.com/TrendpilotAI/Second-Opinion/blob/main/HIPAA_COMPLIANCE.md)

In [ ]:
# ============================================================
# Summary
# ============================================================

print("╔══════════════════════════════════════════════════════════════╗")
print("║     SECOND OPINION — AGENTIC WORKFLOW SUBMISSION            ║")
print("╚══════════════════════════════════════════════════════════════╝")
print()
print(f"✅ MedGemma 4B loaded and run on {len(benchmark_results)} real medical images")
print(f"✅ Average latency: {np.mean(latencies):.1f}s on T4 GPU")
print(f"✅ Average GPU cost: ${np.mean(costs):.4f} per analysis")
print(f"✅ Full pipeline cost: ~${total_pipeline:.3f} per complete analysis")
print(f"✅ Production app: gen-lang-client-0003791133.web.app")
print(f"✅ Open source: github.com/TrendpilotAI/Second-Opinion")
print(f"✅ 40 React components, 4 CI pipelines, HIPAA audit logging")
print()
print("What's real and what's in development:")
print("  ✅ MedGemma inference (demonstrated above)")
print("  ✅ Production web app with full UX")
print("  ✅ Agentic pipeline architecture (pipeline.ts)")
print("  ✅ Medical disclaimers + HIPAA audit logging")
print("  ✅ Modal deployment scripts (infra/modal/)")
print("  🔶 Multi-model consensus (architecture designed, Gemini fallback active)")
print("  🔶 MedSigLIP classification (stubbed, prompt-based classification active)")
print("  🔶 FHIR R4 (typed interfaces, no endpoints yet)")
print()
print("⚠️  DISCLAIMER: This is a demonstration of AI capabilities.")
print("   Not a medical device. Not FDA cleared. Always consult a")
print("   qualified healthcare provider for medical decisions.")
print()
print("Built by Nathan Stevenson | ForwardLane.com | SignalHaus.ai")